# 05. Visualização Espacial da Incerteza (ESDA)

## Contexto Acadêmico e Metodológico
A Análise Exploratória de Dados Espaciais (ESDA - *Exploratory Spatial Data Analysis*) é fundamental para investigar se as incertezas de geocodificação estão distribuídas de forma aleatória no espaço ou se formam aglomerados sistemáticos (clusters). 

Neste notebook, investigamos os registros com **Baixa Certeza de Integração (MCI < 0.5)**, incluindo os órfãos (MCI = 0.0), que representam falhas graves no cadastramento ou no referenciamento espacial (ex: zonas rurais, expansões recentes, aglomerados subnormais). 

Aplicaremos:
1.  **I de Moran Global**: Para mensurar a autocorrelação espacial global do erro posicional.
2.  **LISA (Local Indicators of Spatial Association)**: Para identificar clusters locais estatisticamente significantes (High-High, Low-Low).
3.  **Hexbin Maps**: Para visualização densitária de alta performance em áreas urbanas, mitigando o problema de sombreamento pontual (overplotting).

**Entradas:**
- `data/interim/cnefe_match_bhmap.parquet`


In [ ]:
import sys
import os
from pathlib import Path

os.chdir('..')
sys.path.append(os.getcwd())

%load_ext autoreload
%autoreload 2

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import tempfile
import pysal
import libpysal
from esda.moran import Moran, Moran_Local
from splot.esda import moran_scatterplot, lisa_cluster
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff

from src import config


## 1. Carregamento dos Dados Geoespaciais e Filtragem do Domínio de Incerteza

Vamos focar a análise de autocorrelação e densidade nos registros problemáticos da integração (`MCI < 0.5`).

In [ ]:
temp_dir = Path(tempfile.gettempdir()) / "geocoding_analysis"
match_file = temp_dir / "cnefe_match_bhmap.parquet"

print("Lendo a base espacial cruzada...")
gdf_matched = gpd.read_parquet(match_file)
print(f"Total na base: {len(gdf_matched):,}")

# Definimos "Baixa Certeza" como MCI abaixo de 0.5
low_certainty = gdf_matched[gdf_matched['MCI'] < 0.5].copy()
print(f"Registros de Baixa Incerteza (MCI < 0.5): {len(low_certainty):,}")


## 2. Mapa de Densidade Hexagonal (Hexbin)

Para visualizar a concentração de erros sem sofrer com a sobreposição de milhares de pontos (overplotting), a grelha hexagonal é preferível à interpolação KDE por preservar a estrutura de grade vetorial sem gerar distorções excessivas de suavização.

In [ ]:
# Vamos usar as coordenadas reprojetadas (EPSG:4326) para plotar dinamicamente no mapa.
# Necessitamos remover pontos sem geometria 
low_certainty = low_certainty[~low_certainty.geometry.is_empty & low_certainty.geometry.notna()].copy()

low_cer_4326 = low_certainty.to_crs(epsg=4326)
lon = low_cer_4326.geometry.x
lat = low_cer_4326.geometry.y

fig_hex = ff.create_hexbin_mapbox(
    data_frame=low_cer_4326,
    lat=lat,
    lon=lon,
    nx_hexagon=60,
    opacity=0.7,
    labels={"color": "Contagem de Erros (MCI<0.5)"},
    min_count=5,
    color_continuous_scale="Viridis",
)
fig_hex.update_layout(
    mapbox_style="carto-positron",
    mapbox_zoom=10.5,
    mapbox_center={"lat": -19.91, "lon": -43.94},
    title="Concentração Espacial de Falhas de Integração (Hexbin Map)",
    margin={"r": 0, "t": 40, "l": 0, "b": 0},
    template="plotly_white",
    width=1000, height=700
)
fig_hex.show()

### Interpretação Analítica (Hexbin):
Visualmente, percebemos concentrações periféricas. É necessário comprovar estatisticamente se esta aglomeração é devida ao acaso tipológico ou carrega um agrupamento estrutural, justificando testes ESDA.

## 3. ESDA e Autocorrelação Espacial

A base para esta análise é uma matriz de pesos espaciais que modela a topologia. Para acelerar o cálculo matricial complexo, realizaremos uma amostragem espacial unificada (spatial block sampling / k-nearest neighbors ou distâncias geográficas).

In [ ]:
# Vamos restringir a análise aos dados com erro posicional válido, 
# e usaremos uma amostra estratificada para viabilizar cálculos em tempo real no notebook
valid_dist_gdf = gdf_matched[gdf_matched['spatial_distance'].notna() & (gdf_matched['spatial_distance'] > 0)].copy()
sample_size = min(15000, len(valid_dist_gdf))
esda_sample = valid_dist_gdf.sample(sample_size, random_state=42).copy()

print(f"Tamanho da amostra para matriz de pesos espaciais: {sample_size}")

# 1. Construir Matriz de Pesos Espaciais baseada nos k-vizinhos mais próximos (k=8)
wq = libpysal.weights.KNN.from_dataframe(esda_sample, k=8)
wq.transform = 'r' # row-standardized

y = esda_sample['spatial_distance'].values

# Cálculo do Moran Global (Autocorrelação Espacial)
moran_global = Moran(y, wq)

print("---- I de Moran Global ----")
print(f"I-Value: {moran_global.I:.4f}")
print(f"Expected I: {moran_global.EI:.4f}")
print(f"P-Value (Simulado): {moran_global.p_sim:.4f}")
print(f"Z-Score: {moran_global.z_sim:.4f}")

if moran_global.p_sim < 0.05 and moran_global.I > 0:
    print("\n=> Conclusão: Há autocorrelação espacial POSITIVA significativa. O erro posicional apresenta clusterização.")

## 4. Estatísticas Locais LISA (Local Indicators of Spatial Association)

O diagrama de espalhamento de Moran classifica cada unidade espacial baseada na correlação ponderada de si mesma para seus vizinhos, produzindo quadrantes analíticos: 
* **HH (High-High)**: Alto erro vizinho de Alto erro (Cluster problemático).
* **LL (Low-Low)**: Baixo erro vizinho de Baixo erro (Cluster seguro).
* **HL** / **LH**: Casos anômalos espaciais (Spatial Outliers).

In [ ]:
moran_loc = Moran_Local(y, wq)

# Visualização do Scatterplot do Moran Global/Local
fig, ax = plt.subplots(figsize=(8, 8))
moran_scatterplot(moran_loc, p=0.05, ax=ax)
ax.set_xlabel('Erro Posicional Padronizado', fontsize=12)
ax.set_ylabel('Defasagem Espacial do Erro Posicional', fontsize=12)
ax.set_title(f'Diagrama de Moran (Simulação P<0.05) - I = {moran_global.I:.3f}', fontdict={'fontsize': 14, 'fontweight':'bold', 'family':'Georgia'})
plt.show()

In [ ]:
sns.set_context('notebook')
fig, ax = plt.subplots(figsize=(10, 10))
lisa_cluster(moran_loc, esda_sample, p=0.05, ax=ax, 
             legend_kwds={'loc': 'upper right', 'bbox_to_anchor': (1.05, 1)})
ax.set_title("Mapas de Clusters Espaciais LISA (Erro Posicional)", fontdict={'fontsize': 15, 'fontweight':'bold', 'family':'Georgia'})
plt.tight_layout()
plt.show()

### Conclusões Didáticas (Espaciais)
O mapa **LISA** revela de maneira irrefutável áreas em que a PBH e o IBGE enfrentam dificuldades congênitas de compatibilidade de malha referencial. 
- **Clusters HH (High-High)** são criticidades evidentes. Representam setores contíguos do CNEFE onde desvios topológicos severos se retroalimentam, sinalizando prováveis áreas de urbanidade irregular e falta de arruamento formalizado consolidado.